### Install requirements

In [ ]:
# install libs
!pip install pandas timm scikit-learn

### Test with no pre trained model and no pre processing

In [1]:
import pandas as pd
from pathlib import Path
from timm.data.readers.reader import Reader

# helper function to allow the dataset to work with the timm ImageDataset object
class CsvReader(Reader):
    def __init__(self, img_dir, csv_path, ext=".jpg"):
        super().__init__()
        df = pd.read_csv(csv_path)
        self.img_dir, self.ext = Path(img_dir), ext
        self.ids = df["image_id"].astype(str).str.zfill(5).tolist()  # files are like 00001.jpg and so on
        self.class_to_idx = {c: i for i, c in enumerate(sorted(df["class"].unique()))}  # converts the classes into integer ids
        self.targets = [self.class_to_idx[c] for c in df["class"]]  # creates new column with the classes integer ids

    def __getitem__(self, i):
        # timm takes the open file handle and decodes it with PIL
        return open(self.img_dir / f"{self.ids[i]}{self.ext}", "rb"), self.targets[i]

    def __len__(self):
        return len(self.ids)

    # gets file name
    def _filename(self, index, basename=False, absolute=False):
        return str(self.img_dir / f"{self.ids[index]}{self.ext}")


In [4]:
import time, numpy as np, torch, timm
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from timm.data import ImageDataset, resolve_data_config, create_transform

SUBSET = 8000 # No. of images used from dataset
EPOCHS = 5

reader = CsvReader("dataset/train/train", "dataset/train_labels.csv")
NUM_CLASSES = len(reader.class_to_idx)
class_names = sorted(reader.class_to_idx, key=reader.class_to_idx.get)  # classes ordered by their integer id
targets = np.array(reader.targets)  # integer label per image, used to stratify the splits

print(NUM_CLASSES, class_names)

# Use a non pre trained model to gauge what performence is with no pre processing
model = timm.create_model("resnet50", pretrained=False, num_classes=NUM_CLASSES)
cfg = resolve_data_config({}, model=model)  # use default config for the model

# creates the training and validation dataset
train_full = ImageDataset(root="train/", reader=reader, transform=create_transform(**cfg, is_training=True))
val_full = ImageDataset(root="train/", reader=reader, transform=create_transform(**cfg, is_training=False))

all_idx = np.arange(len(reader))
# cut down the dataset to the 8000 while keeping the class balance (stratifying)
all_idx, _ = train_test_split(all_idx, train_size=SUBSET, stratify=targets, random_state=42)
# split that subset to 80% train and 20% val stratified by class
tr_idx, va_idx = train_test_split(all_idx, test_size=0.2, stratify=targets[all_idx], random_state=42)
print(f"train {len(tr_idx)}, val {len(va_idx)}")

# we have to use num_workers=0 here because of the custom class we're using to make the dataset compatible with timm
# since timm will pickle the dataset object and deploy it to the workers, however that won't work in this case
train_loader = DataLoader(Subset(train_full, tr_idx), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(val_full, va_idx), batch_size=64, num_workers=0)

# use metal on m1 for speed up training
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model.to(device)
opt = torch.optim.AdamW(model.parameters())  # Adam with default params
loss_fn = torch.nn.CrossEntropyLoss()  # standard multi-class classification loss

# Train model
for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()  # put model into training mode
    for x, y in train_loader:
        # move to metal
        x, y = x.to(device), y.to(device)
        opt.zero_grad()  # clear the gradients left over from the previous batch
        loss = loss_fn(model(x), y)  # calc loss
        # backpropagate
        loss.backward()
        # update weights
        opt.step()

    model.eval()  # put model into inference mode for evaluation
    hits = np.zeros(NUM_CLASSES); counts = np.zeros(NUM_CLASSES)  # 2 arrays to keep track of model performance
    # disable gradient tracking, saves on memory
    with torch.no_grad():
        for x, y in val_loader:
            pred = model(x.to(device)).argmax(1).cpu()  # get highest predicted class
            for c in range(NUM_CLASSES):
                m = y == c  # true if current class in loop matches the class for this image
                counts[c] += m.sum().item() # if actual class and class in the loop are the same, add it to the classes count
                hits[c] += (pred[m] == c).sum().item() # if predicted class is correct, add it to the classes' hits
    per_class = hits / np.maximum(counts, 1)  # per-class accuracy (recall), maximum is to not divide by 0

    print(f"epoch {epoch}: last train loss {loss.item():.3f}, val acc {hits.sum()/counts.sum():.3f}, "
          f"balanced acc {per_class.mean():.3f} ({time.time()-t0:.0f}s)")

print("\nper-class val accuracy:")
for n, a, c in zip(class_names, per_class, counts):
    print(f"  {n:10s} {a:.3f}  (n={int(c)})")


8 ['CatEye', 'CatFace', 'CatMouth', 'CatNose', 'DogEye', 'DogFace', 'DogMouth', 'DogNose']
train 6400, val 1600
device: mps
epoch 0: last train loss 1.816, val acc 0.271, balanced acc 0.220 (124s)
epoch 1: last train loss 1.871, val acc 0.287, balanced acc 0.219 (131s)
epoch 2: last train loss 1.972, val acc 0.341, balanced acc 0.289 (131s)
epoch 3: last train loss 1.877, val acc 0.367, balanced acc 0.312 (130s)
epoch 4: last train loss 1.602, val acc 0.410, balanced acc 0.349 (137s)

per-class val accuracy:
  CatEye     0.811  (n=349)
  CatFace    0.345  (n=174)
  CatMouth   0.092  (n=174)
  CatNose    0.069  (n=174)
  DogEye     0.481  (n=291)
  DogFace    0.548  (n=146)
  DogMouth   0.103  (n=146)
  DogNose    0.342  (n=146)


### Step 0: data exploration

In [5]:
import os, pandas as pd, numpy as np
from PIL import Image

test_dir = "dataset/test/test"; tf = sorted(os.listdir(test_dir)); print("test files:", len(tf))  # sorted so the order is the same every run

labels = pd.read_csv("dataset/train_labels.csv")

# analyse each classes' image size and colour mode to see if the model could find a way to cheat
rng = np.random.default_rng(0)  # seeded so the same 600 images are picked every run
rows = []
for i in rng.choice(len(labels), 600, replace=False):
    im = Image.open(f"dataset/train/train/{str(labels.image_id[i]).zfill(5)}.jpg") # gets image data
    rows.append((labels["class"][i], *im.size, im.mode))  # adds image size and color mode to the class
sz = pd.DataFrame(rows, columns=["cls", "w", "h", "mode"]) # convert to df
print(sz[["w","h"]].describe().round(0)); print(sz["mode"].value_counts())
print(sz.groupby("cls")[["w","h"]].mean().round(0))  # average width/height per class

# each class has a unique resolution which could lead to reward hacking, all images are RGB

test files: 29443
           w      h
count  600.0  600.0
mean   117.0  115.0
std    101.0  104.0
min     12.0   12.0
25%     54.0   52.0
50%     84.0   81.0
75%    148.0  138.0
max    964.0  710.0
mode
RGB    600
Name: count, dtype: int64
              w      h
cls                   
CatEye     80.0   80.0
CatFace   213.0  255.0
CatMouth  160.0  112.0
CatNose    78.0   78.0
DogEye     77.0   77.0
DogFace   184.0  184.0
DogMouth  161.0  118.0
DogNose    75.0   75.0


In [ ]:
import hashlib

# helper function to create md5 hashes
def md5(p):
    with open(p, "rb") as f: return hashlib.md5(f.read()).hexdigest()
    
train_hashes = {i: md5(f"dataset/train/train/{str(i).zfill(5)}.jpg") for i in labels.image_id}  # image_id -> hash
test_hases = {f: md5(f"{test_dir}/{f}") for f in tf}  # test file name -> hash

# attach the hash to each image
labels["hash"] = labels.image_id.map(train_hashes)

# check for duplicates (keep=False marks every member of a duplicate group, not just the repeats)
dupes = labels[labels.duplicated("hash", keep=False)]
print(f"train images in duplicate groups: {len(dupes)} ({dupes.hash.nunique()} matches)")

# check if images that have the same hash have conflicting labels
conflict = dupes.groupby("hash")["class"].nunique(); print("groups with conflicting labels:", (conflict > 1).sum())

# check if any of the hashes appear in the test dataset
train_set = set(train_hashes.values()); print("test images also in train (exact bytes):", sum(h in train_set for h in test_hases.values()), "of", len(test_hases))

# full width/height for every training image (PIL only reads the file header for .size, so this is fairly quick)
labels["w"], labels["h_px"] = zip(*[Image.open(f"dataset/train/train/{str(i).zfill(5)}.jpg").size for i in labels.image_id])
print(labels.groupby("class")[["w","h_px"]].agg(["median","min","max"]))

train images in duplicate groups: 30 (15 matches)
groups with conflicting labels: 0
test images also in train (exact bytes): 0 of 29443
              w             h_px          
         median min   max median min   max
class                                     
CatEye     68.0   4   340   68.0   4   340
CatFace   182.0  22   955  215.0  28  1024
CatMouth  114.0   6   578   80.0   4   394
CatNose    68.0   4   340   68.0   4   340
DogEye     62.0  14   886   62.0  14   886
DogFace   155.0  39  2219  155.0  40  2219
DogMouth  104.0  22  1506   77.0  17  1107
DogNose    62.0  14   886   62.0  14   886


In [9]:
from sklearn.metrics import confusion_matrix

# run model eval
model.eval(); P, Y = [], []
with torch.no_grad():
    for x, y in val_loader:
        P.append(model(x.to(device)).argmax(1).cpu()); Y.append(y) # append the actual and predicted class to the arrays

# P = predictions, Y = truth labels
P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()

# create confusion matrix from results
cm = pd.DataFrame(confusion_matrix(Y, P), index=[f"true {c}" for c in class_names], columns=[c[:6] for c in class_names])
print(cm)

sp = lambda c: c.replace("Cat","").replace("Dog",""); an = lambda c: c[:3]  # seperates body part to animal
print("\nanimal accuracy:", np.mean([an(class_names[a]) == an(class_names[b]) for a, b in zip(Y, P)]).round(3)) # acc for animals
print("body part accuracy:", np.mean([sp(class_names[a]) == sp(class_names[b]) for a, b in zip(Y, P)]).round(3)) # acc for body parts

               CatEye  CatFac  CatMou  CatNos  DogEye  DogFac  DogMou  DogNos
true CatEye       283       5       3       7      40       4       0       7
true CatFace       88      60       2       0       4      14       3       3
true CatMouth      74      15      16      16      47       3       1       2
true CatNose       92       4      10      12      51       3       0       2
true DogEye        89       3       1       0     140      15      18      25
true DogFace       34      16       1       0       1      80       9       5
true DogMouth      46       4       1       1      40       8      15      31
true DogNose       39       1       3       1      35       4      13      50

animal accuracy: 0.735
body part accuracy: 0.512


### Step 1: frozen validation split + metrics
One fixed stratified split of the full training set (identical-byte duplicates kept in the same side), saved to `split.csv` so every later experiment is scored on the same images.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, balanced_accuracy_score

lab = labels.rename(columns={"hash": "h"})  # renames hash column to h, helps ensure matching images don't end up on both sides
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)  # fold 0 as val ~= 14%
tr_i, va_i = next(sgkf.split(lab, lab["class"], groups=lab["h"]))  # gets the first fold
split = pd.DataFrame({"image_id": lab.image_id, "class": lab["class"], "is_val": False}) # create df
split.loc[va_i, "is_val"] = True  # mark validation rows
split.to_csv("split.csv", index=False)

# checks there are no duplicates in both datasets using the md5 hash 
print("train", len(tr_i), "val", len(va_i), "| dup groups straddling split:",
      (lab.iloc[tr_i].h.isin(set(lab.iloc[va_i].h)) & lab.iloc[tr_i].duplicated("h", keep=False)).sum())
print(split.groupby("is_val")["class"].value_counts(normalize=True).unstack(0).round(3))  # checks classes are stratified between the training and validation split

# prints out accuracy, balanced accuracy and macro_f1 for training analytics 
def scores(y_true, y_pred):
    return dict(acc=np.mean(y_true == y_pred),
                bal_acc=balanced_accuracy_score(y_true, y_pred),
                macro_f1=f1_score(y_true, y_pred, average="macro"))
print("random-init baseline on the old 1.6k val split:", {k: round(v, 3) for k, v in scores(Y, P).items()})

train 54565 val 9095 | dup groups straddling split: 0
is_val    False  True 
class                 
CatEye    0.218  0.218
CatFace   0.109  0.109
CatMouth  0.109  0.109
CatNose   0.109  0.109
DogEye    0.182  0.182
DogFace   0.091  0.091
DogMouth  0.091  0.091
DogNose   0.091  0.091
random-init baseline on the old 1.6k val split: {'acc': np.float64(0.41), 'bal_acc': 0.349, 'macro_f1': 0.342}


### Step 2: pretrained backbone comparison @112px
Images are cached once as 128x128 uint8 (shorter side resized, centre-cropped to square, so aspect ratio can't leak through stretching). Training uses random 112 crops + flips; val uses the centre 112 crop.

In [15]:
import os, time, numpy as np, pandas as pd, torch, timm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

split = pd.read_csv("split.csv") # gets split from step 1
class_names = sorted(split["class"].unique()); NUM_CLASSES = len(class_names)
y_all = split["class"].map({c: i for i, c in enumerate(class_names)}).to_numpy()  # encode class nmaes as int labels
is_val = split.is_val.to_numpy()

R = 128  # fixed image size
def load_sq(path):
    im = Image.open(path); im.draft("RGB", (R, R)); im = im.convert("RGB")
    w, h = im.size; s = R / min(w, h)  # finds scale factor to reduce image size to R
    im = im.resize((max(R, round(w*s)), max(R, round(h*s))), Image.BICUBIC)  # resizes while not going below R
    w, h = im.size; l, t = (w-R)//2, (h-R)//2  # get top left corner of square image
    return np.asarray(im.crop((l, t, l+R, t+R)))  # crops image

# from stack overflow, save images to cache
CACHE, N = "cache_128.u8", len(split)
if not os.path.exists(CACHE) or os.path.getsize(CACHE) != N*R*R*3:  # build if cache doesn't exist or is incorrect size
    mm = np.memmap(CACHE, dtype=np.uint8, mode="w+", shape=(N, R, R, 3))  # create or override cache
    paths = [f"dataset/train/train/{i:05d}.jpg" for i in split.image_id] # file path for all training images in order
    with ThreadPoolExecutor(8) as ex:  # decode 8 images at a time on separate threads
        for s in range(0, N, 4096):  # processes 4096 images at a time
            mm[s:s+4096] = np.stack(list(ex.map(load_sq, paths[s:s+4096]))) # add cropped images
    mm.flush(); del mm # write to disk
    
X_all = np.memmap(CACHE, dtype=np.uint8, mode="r", shape=(N, R, R, 3))  # reads back cache
print(X_all.shape, ", device", device, ", val", is_val.sum())

(63660, 128, 128, 3) , device mps , val 9095


In [32]:
import threading, traceback
CROP = 112  # lower model input size to stop it from relying on pixel pos 
TRAIN_IDX, VAL_IDX = np.where(~is_val)[0], np.where(is_val)[0]
LOG, RESULTS, STOP = [], {}, threading.Event() # shared state for threads doing background training
def log(m): LOG.append(f"[{time.strftime('%H:%M:%S')}] {m}") # helper function for logging

def make_batch(idx, train, mean, std):
    xb = X_all[idx]  # pull batch from cache
    
    if train: # randomly crops and flips the image
        out = np.empty((len(idx), CROP, CROP, 3), np.uint8)
        oy, ox = np.random.randint(0, R-CROP+1, (2, len(idx))) 
        flip = np.random.rand(len(idx)) < 0.5
        for k in range(len(idx)):
            c = xb[k, oy[k]:oy[k]+CROP, ox[k]:ox[k]+CROP]
            out[k] = c[:, ::-1] if flip[k] else c
    else: # center crop
        o = (R-CROP)//2; out = xb[:, o:o+CROP, o:o+CROP]
        
    # converts cropped batch into a tensor for the model
    x = torch.from_numpy(np.ascontiguousarray(out)).to(device).permute(0, 3, 1, 2).float().div_(255)
    return (x - mean) / std

# evaluates model performance
def evaluate(m, mean, std):
    m.eval(); preds = []
    with torch.no_grad():
        for s in range(0, len(VAL_IDX), 128): # batches of 128, no augmentation
            preds.append(m(make_batch(VAL_IDX[s:s+128], False, mean, std)).argmax(1).cpu())
    return torch.cat(preds).numpy()

def run(tag, name, pretrained, n_train, epochs, lr=3e-4, bs=32, wd=0.05):
    
    torch.manual_seed(0); np.random.seed(0)  # keep the seed the same
    m = timm.create_model(name, pretrained=pretrained, num_classes=NUM_CLASSES).to(device)  # create new model
    # get mean and std per channel pixel values from what the pretrained model expects
    mean = torch.tensor(m.pretrained_cfg["mean"], device=device).view(1, 3, 1, 1)
    std = torch.tensor(m.pretrained_cfg["std"], device=device).view(1, 3, 1, 1)
    
    # use all training images, else use n_train number of images
    tr = TRAIN_IDX if n_train is None else train_test_split(TRAIN_IDX, train_size=n_train, stratify=y_all[TRAIN_IDX], random_state=0)[0]
    spe = len(tr) // bs  # calc steps per epoch
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd) # use the adam optimiser
    # use a LR schedule rather than a fixed value, ramps learning up to mac for the first 15% of the training steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=spe*epochs, pct_start=0.15)
    log(f"{tag} start: {len(tr)} imgs, {spe} steps/epoch, bs {bs}, {epochs} epochs")
    yv = y_all[VAL_IDX]
    
    for ep in range(epochs):
        m.train(); perm = np.random.permutation(tr); tot = 0.0  # put model in training model and shuffle images every epoch
        for s in range(spe): # per step in epoch
            if STOP.is_set(): log(f"{tag} STOPPED"); return # check if thread has been intructed to stop
            idx = np.sort(perm[s*bs:(s+1)*bs])  # grabs next batch
            x, y = make_batch(idx, True, mean, std), torch.from_numpy(y_all[idx]).to(device) # builds batch
            
            # training
            opt.zero_grad() # clear leftover gradients
            loss = torch.nn.functional.cross_entropy(m(x), y) # does a forward pass on the model
            loss.backward() # backpropagation
            opt.step() # update weights
            sched.step() # advance LR schedule
            
            tot += loss.item() # sum up loss
            
            if (s+1) % 200 == 0: log(f"{tag} ep{ep} step {s+1}/{spe} loss {tot/(s+1):.3f}") # progress every 200 steps
            
        p = evaluate(m, mean, std); sc = scores(yv, p)  # score on the validation set after each epoch
        log(f"{tag} epoch {ep}: train loss {tot/spe:.3f} val acc {sc['acc']:.3f} bal_acc {sc['bal_acc']:.3f} macro_f1 {sc['macro_f1']:.3f}")
        
    rec = {c: float(np.mean(p[yv == i] == i)) for i, c in enumerate(class_names)}  # per-class recall after the last epoch
    RESULTS[tag] = dict(**sc, per_class=rec, pred=p)  # saves into shared memory
    torch.save(m.state_dict(), f"ckpt_{tag}.pt")  # save weights
    log(f"{tag} DONE  acc {sc['acc']:.3f}, bal_acc {sc['bal_acc']:.3f}, macro_f1 {sc['macro_f1']:.3f}, saved ckpt_{tag}.pt")
    del m, opt; torch.mps.empty_cache()  # free GPU memory

 # (tag, model, pretrained, n_train or None=all, epochs)
EXPERIMENTS = [("convnext_full", "convnext_tiny", True, None, 5)]

# entry point for training
def job():
    for tag, name, pre, n, ep in EXPERIMENTS:
        try: run(tag, name, pre, n_train=n, epochs=ep)
        except Exception: log(f"{tag} FAILED\n{traceback.format_exc()}")  # log crashes; otherwise the thread would die silently
        if STOP.is_set(): break
    log("ALL DONE")
    
STOP.clear() # stops any threads
threading.Thread(target=job, daemon=True).start() # starts training on threads
print("started")

started


In [38]:
LOG # check training

['[13:45:00] convnext_full start: 54565 imgs, 1705 steps/epoch, bs 32, 5 epochs',
 '[13:45:27] convnext_full ep0 step 1200/1705 loss 0.314',
 '[13:46:52] convnext_full ep4 step 1600/1705 loss 0.104 (7110s)',
 "[13:47:06] val tta=False: {'acc': 0.9534, 'bal_acc': 0.9468, 'macro_f1': 0.9469}",
 "[13:47:49] val tta=True: {'acc': 0.955, 'bal_acc': 0.9487, 'macro_f1': 0.9487}",
 '[13:50:10] wrote submission.csv (29443, 2)',
 "[13:50:10] {'CatEye': 0.216, 'DogEye': 0.189, 'CatNose': 0.115, 'CatFace': 0.106, 'DogNose': 0.097, 'DogFace': 0.095, 'CatMouth': 0.093, 'DogMouth': 0.089}",
 '[13:50:10] SUBMISSION DONE',
 '[13:51:07] convnext_full ep0 step 200/1705 loss 0.489',
 '[13:51:40] convnext_full epoch 4: train loss 0.103 | val acc 0.953 bal_acc 0.946 macro_f1 0.946 (7398s)',
 '[13:51:41] convnext_full DONE  acc 0.953  bal_acc 0.946  macro_f1 0.946  (7399s) | saved ckpt_convnext_full.pt',
 '[13:51:42] ALL DONE',
 '[13:51:42] convnext_full ep0 step 1400/1705 loss 0.307',
 '[13:53:36] convnext_

In [37]:
STOP.set() # stop training

### Step 3: test predictions -> submission.csv
Same preprocessing as validation (shorter side -> 128, centre-crop square, centre 112 crop). Averages logits of the image and its horizontal flip; the validation score with and without the flip is printed as a sanity check that the checkpoint loaded correctly.

In [ ]:
TAG = "convnext_full"
sub = pd.read_csv("dataset/sample_submission.csv")  # gives the required test ids, order and column names
ids = sub.image_id.to_numpy()
t = time.time()
with ThreadPoolExecutor(8) as ex:
    X_te = np.stack(list(ex.map(load_sq, [f"dataset/test/test/{i:05d}.jpg" for i in ids])))
print("test images cached:", X_te.shape, f"{time.time()-t:.0f}s")

m = timm.create_model("convnext_tiny", pretrained=False, num_classes=NUM_CLASSES)  # use model with weights from previous step
m.load_state_dict(torch.load(f"ckpt_{TAG}.pt", map_location="cpu")); m.to(device).eval()  # load on CPU, then move to the GPU
mean = torch.tensor(m.pretrained_cfg["mean"], device=device).view(1, 3, 1, 1)
std = torch.tensor(m.pretrained_cfg["std"], device=device).view(1, 3, 1, 1)
o = (R - CROP) // 2  # offset for the centre crop

@torch.no_grad()  # inference only, so no gradients
def predict_logits(X, tta=True, bs=128):
    out = []
    for s in range(0, len(X), bs):
        xb = np.ascontiguousarray(X[s:s+bs][:, o:o+CROP, o:o+CROP])
        x = (torch.from_numpy(xb).to(device).permute(0, 3, 1, 2).float().div_(255) - mean) / std
        lg = m(x).softmax(1)  # class probabilities (despite the name, these aren't raw logits)
        if tta: lg = lg + m(x.flip(3)).softmax(1)  # add the probabilities for the mirrored image (flip(3) = flip width)
        out.append(lg.cpu())
    return torch.cat(out).numpy()

def finish():   # background thread: long enough to exceed a single tool-call timeout
    try:
        yv = y_all[VAL_IDX]; Xv = X_all[VAL_IDX]
        for tta in (False, True):
            log(f"val tta={tta}: {({k: round(float(v), 4) for k, v in scores(yv, predict_logits(Xv, tta).argmax(1)).items()})}")
        pred = predict_logits(X_te, tta=True).argmax(1)  # most likely class id for each test image
        out = pd.DataFrame({"image_id": ids, "class": np.array(class_names)[pred]})  # class ids -> class names
        out.to_csv("submission.csv", index=False)
        # same columns, row count and ids as the sample file, otherwise Kaggle would reject it
        assert list(out.columns) == list(sub.columns) and len(out) == len(sub) and set(out.image_id) == set(sub.image_id)
        log(f"wrote submission.csv {out.shape}"); log(str(out["class"].value_counts(normalize=True).round(3).to_dict()))  # predicted class mix, to compare with the training mix
    except Exception:
        log("SUBMISSION FAILED\n" + traceback.format_exc())
    log("SUBMISSION DONE")
threading.Thread(target=finish, daemon=True).start(); print("started")

test images cached: (29443, 128, 128, 3) 14s
started


In [ ]:
from sklearn.metrics import confusion_matrix
from PIL import Image

VAL_IDX = np.where(is_val)[0]; yv = y_all[VAL_IDX]  # val rows and their true class ids
res = globals().get("RESULTS", {}).get("convnext_full")

if res is not None:
    p = res["pred"]  # predictions saved by the training run in this session
else:
    net = timm.create_model("convnext_tiny", pretrained=False, num_classes=NUM_CLASSES)
    net.load_state_dict(torch.load("ckpt_convnext_full.pt", map_location="cpu")); net.to(device).eval() # use model with weights from step 2
    mu = torch.tensor(net.pretrained_cfg["mean"], device=device).view(1, 3, 1, 1)
    sd = torch.tensor(net.pretrained_cfg["std"], device=device).view(1, 3, 1, 1)
    crop = 112; o = (R - crop) // 2  # same as CROP in the training-engine cell
    preds = []
    
    with torch.no_grad():
        for s in range(0, len(VAL_IDX), 128):
            xb = np.ascontiguousarray(X_all[VAL_IDX[s:s+128]][:, o:o+crop, o:o+crop])
            x = (torch.from_numpy(xb).to(device).permute(0, 3, 1, 2).float().div_(255) - mu) / sd
            preds.append(net(x).argmax(1).cpu())
            
    p = torch.cat(preds).numpy(); del net
    
cm = pd.DataFrame(confusion_matrix(yv, p), index=[f"true {c}" for c in class_names], columns=[c[:7] for c in class_names])
print(cm); print("\nrecall:", {c: round(float(np.mean(p[yv == i] == i)), 3) for i, c in enumerate(class_names)})

# analysis only (not a model input): are errors concentrated in particular crop shapes?
va = split.iloc[VAL_IDX].reset_index(drop=True)  # validation rows of split.csv, in the same order as p
wh = np.array([Image.open(f"dataset/train/train/{i:05d}.jpg").size for i in va.image_id])  # original (width, height) of each val image
va["w"], va["h"], va["aspect"] = wh[:, 0], wh[:, 1], wh[:, 0] / wh[:, 1]
va["short"] = wh.min(1); va["ok"] = (p == yv)  # shorter side in px; whether the prediction was correct

for c in ["CatMouth", "CatNose"]:
    d = va[va["class"] == c].copy()
    d["aspect_bin"] = pd.qcut(d.aspect, 4, duplicates="drop"); d["size_bin"] = pd.qcut(d.short, 4, duplicates="drop")  # 4 equal-count bins
    # in the tables below, mean = accuracy within the bin and size = number of images in it
    print(f"\n{c}: accuracy by aspect ratio (w/h) quartile"); print(d.groupby("aspect_bin", observed=True).ok.agg(["mean", "size"]).round(3))
    print(f"{c}: accuracy by shorter-side quartile (px)"); print(d.groupby("size_bin", observed=True).ok.agg(["mean", "size"]).round(3))

               CatEye  CatFace  CatMout  CatNose  DogEye  DogFace  DogMout  \
true CatEye      1975        1        1        1       6        0        0   
true CatFace        0      991        0        0       1        0        0   
true CatMouth       3        0      767      222       0        0        0   
true CatNose        2        0      152      836       2        0        0   
true DogEye         6        0        0        0    1647        0        0   
true DogFace        0        1        0        0       0      825        1   
true DogMouth       0        0        0        0       0        0      810   
true DogNose        1        0        0        0       0        0        6   

               DogNose  
true CatEye          0  
true CatFace         0  
true CatMouth        0  
true CatNose         0  
true DogEye          1  
true DogFace         0  
true DogMouth       17  
true DogNose       820  

recall: {'CatEye': 0.995, 'CatFace': 0.999, 'CatMouth': 0.773, 'CatNose